In [0]:
# ==========================================================
# 04_validacao_cruzada
# Camada: Validação
# Objetivo: provar consistência entre as Golds após a
# troca de fonte (metas -> municipio.parquet)
# ==========================================================

from pyspark.sql import functions as F

CATALOGO = "dbw_fiap_techchallenge_prod_7405607602465611"
SCHEMA   = "default"

T_GOLD_MUNICIPIO = f"{CATALOGO}.{SCHEMA}.gold_indicador_municipio"
T_GOLD_UF        = f"{CATALOGO}.{SCHEMA}.gold_indicador_uf"
T_GOLD_ALUNOS    = f"{CATALOGO}.{SCHEMA}.gold_alunos"

print("Tabelas de validação:")
for t in [T_GOLD_MUNICIPIO, T_GOLD_UF, T_GOLD_ALUNOS]:
    print(f"  {t}")

In [0]:
# ==========================================================
# Célula 2 — Inventário das Golds
# ==========================================================

tabelas = {
    "gold_indicador_municipio": T_GOLD_MUNICIPIO,
    "gold_indicador_uf":        T_GOLD_UF,
    "gold_alunos":              T_GOLD_ALUNOS,
}

print("=" * 62)
print("INVENTÁRIO DAS GOLDS")
print("=" * 62)

for nome, full in tabelas.items():
    try:
        df = spark.table(full)
        print(f"{nome:<28} {df.count():>10,} registros | {len(df.columns):>2} colunas")
    except Exception as e:
        print(f"{nome:<28} ERRO: {e}")

print("=" * 62)

In [0]:
# ==========================================================
# Célula 3 — Cobertura cruzada: alunos x indicador_municipio
# ==========================================================

df_alunos = spark.table(T_GOLD_ALUNOS)
df_mun    = spark.table(T_GOLD_MUNICIPIO)

mun_alunos = {r[0] for r in df_alunos.select("id_municipio").distinct().collect()}
mun_ind    = {r[0] for r in df_mun.select("id_municipio").distinct().collect()}

print("=" * 62)
print("COBERTURA CRUZADA — MUNICÍPIOS")
print("=" * 62)
print(f"Municípios em gold_alunos             : {len(mun_alunos):,}")
print(f"Municípios em gold_indicador_municipio: {len(mun_ind):,}")
print(f"Interseção                            : {len(mun_alunos & mun_ind):,}")
print(f"Só em gold_alunos                     : {len(mun_alunos - mun_ind):,}")
print(f"Só em gold_indicador_municipio        : {len(mun_ind - mun_alunos):,}")
print("=" * 62)

In [0]:
# ==========================================================
# Célula 4 — Taxa nacional ponderada e cobertura de UFs
# gold_alunos já traz taxa_alfabetizacao pronta por município
# ==========================================================

df_uf    = spark.table(T_GOLD_UF)
df_aluno = spark.table(T_GOLD_ALUNOS)

print("=" * 70)
print("CONSISTÊNCIA UF -> BRASIL")
print("=" * 70)

# 1) Taxa nacional ponderada a partir da gold_alunos
nacional = (df_aluno
    .groupBy("ano")
    .agg(
        F.sum("qtd_alunos").alias("alunos"),
        F.sum(F.col("taxa_alfabetizacao") * F.col("qtd_alunos")).alias("alfab_ponderado"),
    )
    .withColumn("taxa_nacional",
        F.round(F.col("alfab_ponderado") / F.col("alunos"), 2))
    .select("ano", "alunos", "taxa_nacional")
    .orderBy("ano"))

print("\n[1] Taxa nacional ponderada (gold_alunos):")
nacional.show(truncate=False)

# 2) Cobertura de UFs por ano
cobertura = (df_uf
    .groupBy("ano")
    .agg(
        F.count("*").alias("ufs_total"),
        F.sum(F.when(F.col("taxa_observada").isNotNull(), 1).otherwise(0)).alias("ufs_com_dado"),
        F.sum(F.when(F.col("status_meta") == "sem_dado", 1).otherwise(0)).alias("ufs_sem_dado"),
    )
    .orderBy("ano"))

print("\n[2] Cobertura de UFs por ano:")
cobertura.show(truncate=False)

# 3) Distribuição de status_meta
print("\n[3] Distribuição de status_meta:")
(df_uf.groupBy("ano", "status_meta")
    .count()
    .orderBy("ano", "status_meta")
    .show(truncate=False))

print("=" * 70)

In [0]:
# ==========================================================
# Célula 5 — Investigar sem_meta em 2023
# ==========================================================

df_uf = spark.table(T_GOLD_UF)

print("=" * 70)
print("INVESTIGAÇÃO: sem_meta em 2023")
print("=" * 70)

print("\n[1] UFs com status sem_meta em 2023:")
(df_uf
    .filter((F.col("ano") == 2023) & (F.col("status_meta") == "sem_meta"))
    .select("sigla_uf", "taxa_observada", "meta_do_ano", "gap_pp")
    .orderBy("sigla_uf")
    .show(30, truncate=False))

print("\n[2] UFs com meta_do_ano preenchida em 2023:")
(df_uf
    .filter((F.col("ano") == 2023) & F.col("meta_do_ano").isNotNull())
    .select("sigla_uf", "taxa_observada", "meta_do_ano", "status_meta")
    .orderBy("sigla_uf")
    .show(30, truncate=False))

print("\n[3] Amostra de metas por ano:")
(df_uf
    .select("ano", "sigla_uf", "meta_do_ano", "meta_2024", "meta_2025", "status_meta")
    .filter(F.col("sigla_uf").isin("SP", "RJ", "MG"))
    .orderBy("sigla_uf", "ano")
    .show(truncate=False))

print("=" * 70)

In [0]:
# ==========================================================
# Célula 6 — Confirmar ausência de meta em 2023
# ==========================================================

df_uf = spark.table(T_GOLD_UF)

print("=" * 70)
print("CONFIRMAÇÃO: base de metas começa em 2024")
print("=" * 70)

print("\n[1] Cobertura de meta_do_ano por ano:")
(df_uf
    .groupBy("ano")
    .agg(
        F.count("*").alias("ufs"),
        F.sum(F.when(F.col("meta_do_ano").isNotNull(), 1).otherwise(0)).alias("com_meta"),
        F.sum(F.when(F.col("meta_do_ano").isNull(), 1).otherwise(0)).alias("sem_meta"),
    )
    .orderBy("ano")
    .show(truncate=False))

print("\n[2] meta_2024 preenchida por ano:")
(df_uf
    .groupBy("ano")
    .agg(
        F.sum(F.when(F.col("meta_2024").isNotNull(), 1).otherwise(0)).alias("com_meta_2024"),
    )
    .orderBy("ano")
    .show(truncate=False))

print("\n[3] Resumo da validação:")
print("  - 2023: linha de base, sem meta publicada (esperado)")
print("  - 2024: primeiro ano com meta comparável")
print("  - 2025: meta futura, sem dado observado (esperado)")

print("=" * 70)

In [0]:
# ==========================================================
# Célula 7 — Veredito final da validação cruzada
# ==========================================================

print("=" * 70)
print("VEREDITO FINAL — VALIDAÇÃO CRUZADA")
print("=" * 70)

checks = [
    ("Integridade referencial (municípios órfãos)",
     "PASS", "0 municípios em gold_alunos ausentes em gold_indicador_municipio"),
    ("Cobertura de municípios",
     "PASS", "5.547 de 5.550 (99,95%)"),
    ("Taxa nacional 2023",
     "PASS", "57,89% sobre 1.503.058 alunos"),
    ("Taxa nacional 2024",
     "PASS", "59,57% sobre 1.852.788 alunos"),
    ("Meta 2023",
     "N/A",  "Base de metas do INEP começa em 2024 (linha de base)"),
    ("Meta 2024",
     "PASS", "24 UFs comparadas: 12 abaixo, 12 atingiu"),
    ("Meta 2025",
     "N/A",  "Ano futuro, sem dado observado"),
]

for nome, status, detalhe in checks:
    print(f"[{status:<4}] {nome}")
    print(f"         {detalhe}")

print("=" * 70)
print("RESULTADO: pipeline consistente. Nenhuma falha crítica.")
print("=" * 70)